# D11 -- 基因调控网络 (Gene Regulatory Network via pySCENIC)

本 notebook 做基因调控网络推断，通过 pySCENIC 三步流程识别细胞类型
特异的转录调控因子 (regulon) 及其靶基因：

1. **GRNBoost2** (arboreto) -- 基于共表达的 TF-靶基因推断，**不需数据库**
2. **cisTarget** (pyscenic ctx) -- 基于顺式调控元件基序的模块修剪，**需数据库 + 环境兼容**
3. **AUCell** (pyscenic aucell) -- 每细胞 regulon 活性打分，**依赖 cisTarget 输出**

## 生物学背景

**为什么做 GRN？** 差异表达分析告诉我们"哪些基因变了"，
基因调控网络告诉我们"谁在驱动这些变化"。TF（转录因子）是基因表达
调控的核心执行者——找到关键的调控因子（如胃上皮分化主控因子
GATA4/GATA6/CDX2 等），比单独看每个差异基因更有生物学解释力。

**为什么主控转录因子是下游分析的核心？** 一个细胞类型的身份
由少数几个主控转录因子 (master TF) 决定——它们调控数百个下游
靶基因，构成转录调控层级 (regulatory hierarchy) 的顶端。
找到这些 master TF 并量化其活性，是理解细胞状态转变（如
胃癌前病变→早癌的恶性转化）的核心入口。

**为什么用 pySCENIC？** SCENIC (Single-Cell rEgulatory Network
Inference and Clustering) 是在单细胞领域经过充分验证的 GRN 方法，
发表于 Nature Methods (2017)。pySCENIC 是其 Python 实现，
三步流程设计清晰——先推断共表达网络，再用顺式调控数据库做"生物学
真实性"过滤，最后对每个细胞打分。

## pySCENIC 三步流程详解

### 第一步：GRNBoost2（共表达推断）
对每个靶基因，用 Gradient Boosting 回归找出哪些 TF 的表达量能预测
该基因的表达量。**此步骤是纯机器学习方法，不需要外部数据库。**
需要 dask 进行分布式计算——本 notebook 启动本地 dask 集群。

### 第二步：cisTarget（模块修剪）
对每个 TF，检查其共表达靶基因的调控区是否富集该 TF 的 DNA 结合基序
(motif)。只有那些在基因组序列上有"结构基础"的调控链接才保留为
regulon。**此步骤需要物种特异的 cisTarget 数据库**：
- ranking 数据库 (.feather, ~1-2GB)
- motif 注释文件 (.tbl)
- 物种 TF 列表 (.txt)

**额外依赖**：pyscenic 0.12.1 的 cisTarget 模块使用 `np.object`，
在 numpy >= 1.24 中已被移除。需降级 numpy（`pip install numpy<1.24`）
或应用 patch。见下方守卫提示。

### 第三步：AUCell（活性打分）
对每个细胞，计算其表达的基因在多大程度上富集于每个 regulon 的
靶基因集合。产出每细胞 x regulon 的活性矩阵——可用于下游聚类、
可视化、跨条件比较。

## 数据库依赖与当前状态

GRNBoost2 步骤不需要数据库，可以**直接运行**。
cisTarget 和 AUCell 步骤需要 (1) hg38 cisTarget 数据库 和
(2) numpy 兼容性。若条件不满足，这些步骤会优雅跳过并给出指引。

产出：
- GRNBoost2 共表达网络 → `results/tables/13_grnboost2_links.csv`
- regulon 活性矩阵 (AUCell) → `adata.obsm['X_regulon_auc']`
- 可视化 → `results/figures/13_grn_*`


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **下游**：下游分析（regulon 活性聚类 / 主控 TF 鉴定），产出 `13_grn_v*.h5ad` + GRNBoost2 共表达网络 + regulon AUC 矩阵

### 为什么要迭代回跑？
基因调控网络 (GRN) 的结果是下游分析和 PI 生物学判断的基础。如果在后续分析中发现：
- DEG 阈值过高导致遗漏关键基因、过低导致假阳性
- 通路富集缺少预期应出现的生物学通路
- 调控网络缺少已知的主控转录因子
- CNV 信号不符合病理学预期
可能需要调整本 stage 的参数重新计算。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`（旧版不覆盖）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改以下参数后重跑本 notebook）：
- `N_HVG` — GRNBoost2 使用的 HVG 基因数
- `DASK_N_WORKERS` — dask 本地集群线程数
- `CISTARGET_NUM_WORKERS` / `AUC_NUM_WORKERS` — 并行度
- cisTarget 数据库切换（换物种或 motif 版本时修改 PARAMS 中的三个路径）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：下游 notebook 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"13_grn"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询「基因调控网络 (GRN) 有哪些版本？哪些依赖 06_annotated_v1？」，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH              -- 06 注释结果 h5ad
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns['version'] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# LEIDEN_COL                 -- Leiden 簇标签 obs 列
# N_HVG                      -- GRNBoost2 取 top HVG 基因数（控制规模/运行时间）
#                               真实分析建议 >=2000，快速验证可降至 500-1000
# DASK_N_WORKERS             -- dask 本地集群工作线程数
# CISTARGET_RANKINGS_PATH    -- cisTarget 排名数据库 .feather 路径
# CISTARGET_MOTIF_PATH       -- cisTarget motif 注释 .tbl 路径
# CISTARGET_TF_LIST_PATH     -- 物种 TF 列表 .txt 路径
#                               人类推荐：Lambert et al. 2018 (Cell) curated TF list
# CISTARGET_NUM_WORKERS      -- cisTarget 并行度
# AUC_NUM_WORKERS            -- AUCell 并行度

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_PATH   = "results/13_grn_v1.h5ad"

LEIDEN_COL = "leiden_res_0.6"

# GRNBoost2 规模控制：HVG 子集基因数。数据集 >10K 基因时建议限 1500-2000。
N_HVG = 2000
DASK_N_WORKERS = 4

# ---- cisTarget 数据库路径（PI 下载后修改以下路径） ----
# 推荐下载地址（hg38 人类数据库）:
#   https://resources.aertslab.org/cistarget/
# 需要三个文件：
#   1. Ranking DB (.feather): hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather
#   2. Motif 注释 (.tbl):    motifs-v9-nr.hgnc-m0.001-o0.0.tbl
#   3. TF 列表 (.txt):       allTFs_hg38.txt
# 放置位置：项目目录 references/cisTarget/ 下。
CISTARGET_RANKINGS_PATH = "references/cisTarget/hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather"
CISTARGET_MOTIF_PATH    = "references/cisTarget/motifs-v9-nr.hgnc-m0.001-o0.0.tbl"
CISTARGET_TF_LIST_PATH  = "references/cisTarget/allTFs_hg38.txt"
CISTARGET_NUM_WORKERS   = 4

AUC_NUM_WORKERS = 4


In [ ]:
# === setup：sys.path + 导入 + 加载上游 ===

# 1. 确保框架 src/ 在 sys.path 并切换到项目根目录
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("references/cisTarget", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")


# 2. 导入依赖
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# pySCENIC 所需：
# arboreto: GRNBoost2 共表达推断引擎（不需数据库，无 numpy 兼容问题）
from arboreto.core import create_graph, SGBM_KWARGS
# dask 分布式计算（GRNBoost2 底层依赖）
from distributed import Client, LocalCluster

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="anndata")
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")


# 3. 加载上游 06 输出
# 契约：需包含 highly_variable 列（04 产出）和 LEIDEN_COL。
# 加载上游 06 输出。
# 契约：需包含 highly_variable 列（04 产出）和 LEIDEN_COL。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 检查 HVG
if "highly_variable" in adata.var.columns:
    _n_hvg = adata.var["highly_variable"].sum()
    print(f"HVG 数: {_n_hvg}")
else:
    _n_hvg = 0
    print("WARNING: adata.var 无 'highly_variable' 列，将使用所有基因——运行时间会显著延长！")
    print("  建议先在 04 中跑 sc.pp.highly_variable_genes() 再重跑本 notebook。")

# 检查 obs 列
print(f"LEIDEN_COL '{LEIDEN_COL}' 存在: {LEIDEN_COL in adata.obs.columns}")
if LEIDEN_COL in adata.obs.columns:
    print(f"  簇数: {adata.obs[LEIDEN_COL].nunique()}")

# 检查 layers / obsm
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")


## 数据库与环境守卫

在运行 pySCENIC 之前，检查：
1. **cisTarget 数据库文件** 是否就绪（ranking .feather + motif .tbl + TF list .txt）
2. **pyscenic 模块可导入性**（numpy 兼容性检查——pyscenic 0.12.1 使用
   已废弃的 `np.object`，在 numpy >= 1.24 中会触发 ImportError）

任一条件不满足，cisTarget 和 AUCell 步骤将优雅跳过——
只跑 GRNBoost2（不需要数据库且无 numpy 兼容问题）。

### numpy 兼容性说明

pyscenic 0.12.1 的 `transform.py` 使用了 `np.object`，该别名在
numpy >= 1.24 中已移除。受影响模块：`pyscenic.prune`（cisTarget 修剪）、
`pyscenic.cli.utils`（motif 加载）。

**修复方式（二选一）**：
- **降级 numpy**：`pip install "numpy<1.24"`（简单但可能影响其他依赖）
- **热修复 pyscenic**：将 `pyscenic/transform.py` 第 42 行的 `np.object`
  替换为 `object`（不影响功能，三行改动）

修复后重跑本 notebook 即可启用 cisTarget + AUCell。


In [ ]:
# === 数据库 + 环境兼容性检查 ===
import os as _os

# ---- 1. 数据库文件检查 ----
_db_checks = {
    "ranking_db":  (CISTARGET_RANKINGS_PATH, "cisTarget ranking 数据库 (.feather)"),
    "motif_annot": (CISTARGET_MOTIF_PATH,    "motif 注释文件 (.tbl)"),
    "tf_list":     (CISTARGET_TF_LIST_PATH,  "TF 列表 (.txt)"),
}

_cistarget_db_ready = True
for _key, (_path, _desc) in _db_checks.items():
    _exists = _os.path.exists(_path)
    _size = _os.path.getsize(_path) if _exists else 0
    _status = f"OK ({_size:,} bytes)" if _exists else "MISSING"
    print(f"  [{_status}] {_desc}: {_path}")
    if not _exists:
        _cistarget_db_ready = False

# ---- 2. pyscenic 模块导入检查（numpy 兼容性）——无条件执行 ----
# 数据库缺失时也检查 numpy 兼容性，避免 PI 下完 1.5G 数据库后
# 重跑才发现 numpy 2.x np.object 不兼容——两步循环。
# 注意：pyscenic 0.12.1 transform.py 的 np.object 触发 AttributeError
# （而非 ImportError），因为模块级代码执行时 numpy.__getattr__ 抛出异常。
#
# pyscenic 0.12.1 的 transform.py 使用已废弃的 np.object（numpy>=1.24 已移除）。
# 在导入前做 monkey-patch，不改动 pyscenic 源码（保护原 Mac 环境兼容性）。
if not hasattr(np, 'object'):
    np.object = object  # pyscenic transform.py 需要 np.object 别名

_cistarget_importable = False
_cistarget_import_error = ""
try:
    from pyscenic.prune import prune2df
    from pyscenic.cli.utils import load_signatures
    from pyscenic.aucell import aucell
    _cistarget_importable = True
    print("  pyscenic cisTarget 模块可导入: OK")
except Exception as _e:
    _cistarget_import_error = str(_e)
    print(f"  pyscenic cisTarget 模块不可导入: {_e}")

# ---- 3. 综合判断 ----
_cistarget_ready = _cistarget_db_ready and _cistarget_importable

if not _cistarget_db_ready and not _cistarget_importable:
    _skip_msg = (
        "=" * 60 + "\n"
        "cisTarget 数据库不完整 且 numpy 不兼容——"
        "GRNBoost2 可正常运行，cisTarget 和 AUCell 步骤将跳过。\n\n"
        "需同时满足两个条件才可启用完整 pySCENIC 三步流程:\n"
        "  [1] 下载 hg38 cisTarget 数据库 (>1.5GB):\n"
        "      https://resources.aertslab.org/cistarget/\n"
        f"      - {CISTARGET_RANKINGS_PATH}\n"
        f"      - {CISTARGET_MOTIF_PATH}\n"
        f"      - {CISTARGET_TF_LIST_PATH}\n"
        f"  [2] 修复 numpy 兼容性: {_cistarget_import_error}\n"
        "      修复方式（二选一）:\n"
        "        a. 降级 numpy: pip install 'numpy<1.24'\n"
        "        b. 热修复 pyscenic: 将 pyscenic/transform.py 第 42 行的\n"
        "           np.object 替换为 object\n"
        "两项均修复后重跑本 notebook 即可。\n"
        "=" * 60
    )
    print(_skip_msg)
elif not _cistarget_db_ready:
    _skip_msg = (
        "=" * 60 + "\n"
        "cisTarget 数据库不完整——GRNBoost2 可正常运行，"
        "cisTarget 和 AUCell 步骤将跳过。\n\n"
        "要启用完整的 pySCENIC 三步流程，请下载 hg38 cisTarget 数据库:\n"
        "  https://resources.aertslab.org/cistarget/\n\n"
        "需要三个文件（放置在项目根目录的 references/cisTarget/ 下）:\n"
        f"  1. {CISTARGET_RANKINGS_PATH}\n"
        f"  2. {CISTARGET_MOTIF_PATH}\n"
        f"  3. {CISTARGET_TF_LIST_PATH}\n\n"
        "下载完成后重跑本 notebook 即可。\n"
        "=" * 60
    )
    print(_skip_msg)
elif not _cistarget_importable:
    _skip_msg = (
        "=" * 60 + "\n"
        "cisTarget 数据库已就绪，但 pyscenic cisTarget 模块因 numpy "
        "兼容性无法导入。GRNBoost2 可正常运行。\n\n"
        f"错误详情: {_cistarget_import_error}\n\n"
        "修复方式（二选一）:\n"
        "  1. 降级 numpy: pip install 'numpy<1.24'\n"
        "  2. 热修复 pyscenic: 将 pyscenic/transform.py 第 42 行的\n"
        "     np.object 替换为 object\n"
        "     (位于 conda env 的 site-packages/pyscenic/transform.py)\n\n"
        "修复后重跑本 notebook 即可。\n"
        "=" * 60
    )
    print(_skip_msg)
else:
    print("所有 cisTarget 数据库就绪且 pyscenic 模块可导入——将完整运行 pySCENIC 三步流程。")

## 1. GRNBoost2 -- TF-靶基因共表达推断

**为什么先做共表达推断？** 这是 pySCENIC 的第一步——在所有基因中
识别 TF-target 候选调控关系。GRNBoost2 对每个靶基因训练一个
Gradient Boosting 回归模型，用所有 TF 的表达量来预测该靶基因的表达量。
预测能力强意味着该 TF 与靶基因之间存在共表达关系，可能是直接调控。

**TF-target 的方向性**：GRNBoost2 识别的是有方向的调控关系
（TF → target），而非无向的共表达。这是 GRN 与 WGCNA 等共表达
网络分析的关键区别——GRN 明确区分"调控者"与"被调控者"，
更适合回答"谁在驱动这个生物学过程"。

**为什么用 HVG 子集？** 完整基因集（~38K 基因）的 GRNBoost2 运行
时间与基因数的平方成正比，全量运行需要数小时。取 top HVG（高变异基因）
子集可以：1) 过滤掉信息量低的基因，2) 将运行时间控制在分钟级。
在真实分析中，建议使用 HVG 2000-4000 的规模。

**为什么需要 counts 矩阵？** GRNBoost2 本质是回归模型，需要原始计数
而非 log-normalized 值。log 变换会压缩方差，削弱回归模型的预测信号。

**arboreto / dask 实现说明**：本 cell 直接调用 `arboreto.core.create_graph`
而非高层的 `grnboost2()` wrapper，因为 arboreto 0.12.1 的 `diy()`
使用 `include_meta=False`——这在 dask >= 2024 中会触发
`from_delayed([])` TypeError。改用 `include_meta=True` 是干净且
教学透明的工作方案（见 cell 代码注释）。

### GRNBoost2 可视化（紧随计算产出）

GRNBoost2 产出所有 TF-target 链接的重要性评分。通过以下两图快速评估
共表达网络的质量和结构：

- **重要性分布直方图**：展示所有 TF-target links 的 importance 分布，
  峰度和尾部反映网络中强/弱链接的比例
- **TF 出度条形图**：每个 TF 调控的靶基因数量排名，
  高出度 TF 可能是全局调控枢纽


In [ ]:
# === Step 1: GRNBoost2 共表达网络推断 ===
# 使用 HVG 子集控制规模。对每个靶基因，用所有基因（作为候选 TF）
# 的表达量来训练 Gradient Boosting 回归预测模型。
# 重要性得分越高 -> TF 对该靶基因的表达预测力越强 -> 可能是调控关系。

# 1a. 选取 HVG 子集
_n_hvg_use = min(N_HVG, _n_hvg if _n_hvg > 0 else adata.n_vars)
if "highly_variable" in adata.var.columns:
    _hvg_mask = adata.var["highly_variable"].values
    _hvg_genes = adata.var_names[_hvg_mask][:_n_hvg_use].tolist()
else:
    _hvg_genes = adata.var_names[:_n_hvg_use].tolist()

print(f"GRNBoost2 使用 {len(_hvg_genes)} 个 HVG 基因"
      f"（共 {adata.n_vars} 个基因）")

# 1b. 构建表达矩阵：优先 counts layer（原始计数），其次 adata.raw，最后 adata.X
if "counts" in adata.layers:
    _X_grn = adata[:, _hvg_genes].layers["counts"]
elif adata.raw is not None:
    _X_grn = adata.raw[:, _hvg_genes].X
else:
    _X_grn = adata[:, _hvg_genes].X

# 转为 dense numpy（GRNBoost2 需要 dense 矩阵做回归）
if sp.issparse(_X_grn):
    _X_grn = _X_grn.toarray()
else:
    _X_grn = np.asarray(_X_grn)
print(f"表达矩阵: {_X_grn.shape[0]} 细胞 x {_X_grn.shape[1]} 基因  "
      f"(内存 {_X_grn.nbytes / 1e6:.1f} MB)")

# 1c. 启动 dask 本地集群
# 为什么用 dask？GRNBoost2 对每个靶基因独立训练回归模型——是天生的
# embarrassingly parallel 问题。dask 将任务分发到多线程 worker 并行执行。
print(f"启动 dask LocalCluster (n_workers={DASK_N_WORKERS})...")
_local_cluster = LocalCluster(
    n_workers=DASK_N_WORKERS,
    threads_per_worker=1,
    memory_limit="4GB",
)
_client = Client(_local_cluster)
print(f"dask dashboard: {_client.dashboard_link}")

# 1d. 构建计算图（include_meta=True 避开 dask >= 2024 兼容问题）
_graph = create_graph(
    expression_matrix=_X_grn,
    gene_names=_hvg_genes,
    tf_names=_hvg_genes,            # 所有 HVG 基因都视为候选 TF
    regressor_type="GBM",           # Gradient Boosting
    regressor_kwargs=SGBM_KWARGS,   # 默认超参：learning_rate=0.01, n_estimators=5000
    client=_client,
    target_genes="all",
    include_meta=True,              # 必须 True——dask >= 2024 兼容性
    early_stop_window_length=25,
    limit=None,                     # 取所有链接，后续按 importance 排序过滤
    seed=42,
)
print(f"dask 计算图已构建: {_graph[0].npartitions} partitions")

# 1e. 执行计算
print("正在执行 GRNBoost2 (Gradient Boosting 在 dask workers 上并行)...")
_links_df, _meta_df = _client.compute(_graph, sync=True)
_links_df = _links_df.sort_values("importance", ascending=False)
print(f"GRNBoost2 完成: {len(_links_df):,} 条 TF-target 链接")

# 释放 dask 资源
_client.close()
_local_cluster.close()
del _X_grn, _graph, _client, _local_cluster
gc.collect()
print("dask 集群已关闭，内存已释放")


In [ ]:
# GRNBoost2 结果概览。
print(f"TF-target 链接总数: {len(_links_df):,}")
print(f"重要性范围: [{_links_df['importance'].min():.4f}, {_links_df['importance'].max():.4f}]")
print(f"TF 种类数: {_links_df['TF'].nunique()}")
print(f"靶基因种类数: {_links_df['target'].nunique()}")
print()

# Top 20 高重要性链接
print("Top 20 TF-target 调控链接（按 importance 降序）:")
print(_links_df.head(20)[["TF", "target", "importance"]].to_string(index=False))
print()

# 最"活跃"的 TF——出度最高 / 平均重要性最高的 TF
_tf_stats = _links_df.groupby("TF").agg(
    n_targets=("target", "nunique"),
    mean_importance=("importance", "mean"),
).sort_values("n_targets", ascending=False)
print("Top 10 TF（按靶基因出度排序）:")
print(_tf_stats.head(10).to_string())

# 保存 GRNBoost2 全量结果
_grn_csv = "results/tables/13_grnboost2_links.csv"
_links_df.to_csv(_grn_csv, index=False)
print(f"\nGRNBoost2 共表达网络已保存: {_grn_csv} ({len(_links_df):,} 行)")

# Top 500 高重要性链接——精炼版方便 PI 快速查看
_top500 = _links_df.head(500)
_top500_csv = "results/tables/13_grnboost2_top500.csv"
_top500.to_csv(_top500_csv, index=False)
print(f"Top 500 链接已保存: {_top500_csv}")


In [ ]:
# === GRNBoost2 共表达网络可视化 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fig 1a: Importance 分布直方图
axes[0].hist(_links_df["importance"], bins=100, color="#1f77b4",
             alpha=0.7, edgecolor="white", linewidth=0.3)
_q95 = _links_df["importance"].quantile(0.95)
axes[0].axvline(_q95, color="red", ls="--", lw=1.2,
                label=f"95% quantile ({_q95:.4f})")
axes[0].set_xlabel("Importance")
axes[0].set_ylabel("TF-target Link Count")
axes[0].set_title("GRNBoost2: TF-Target Link Importance Distribution")
axes[0].legend(loc="upper right", frameon=False)

# Fig 1b: Top 15 TF 出度（调控靶基因数排名）
_tf_degree = _links_df.groupby("TF").size().sort_values(ascending=False).head(15)
axes[1].barh(range(len(_tf_degree)), _tf_degree.values[::-1],
             color="#ff7f0e", edgecolor="white", linewidth=0.5)
axes[1].set_yticks(range(len(_tf_degree)))
axes[1].set_yticklabels(_tf_degree.index[::-1], fontsize=8)
axes[1].set_xlabel("Number of Target Genes")
axes[1].set_title("Top 15 TFs by Target Gene Count (out-degree)")
axes[1].invert_yaxis()

plt.tight_layout()
_grn_fig1 = "results/figures/13_grnboost2_summary.png"
fig.savefig(_grn_fig1, dpi=200, bbox_inches="tight")
plt.close("all")
print(f"GRNBoost2 概览图已保存: {_grn_fig1}")


## 2. cisTarget -- 顺式调控模块修剪

**为什么需要 cisTarget？** GRNBoost2 找到的是纯计算层面的"共表达"关
系——A 和 B 总是在一起高表达，但 A 可能并不直接调控 B（可能是被第三
个基因 C 驱动，或只是技术噪音）。cisTarget 通过检查"调控区 DNA 序列上
是否有 TF 的结合基序"来做生物学修剪——只有那些在基因组上有结构基础的
调控关系才保留。

**为什么基因组层面的验证不可或缺？** 共表达可能来自多种原因
（共享的 enhancer、染色质状态、技术批次效应），而 DNA 结合基序
的富集提供了"该 TF 蛋白确实能物理结合到这些靶基因的调控区"的结构证据。
这一步将统计推断转化为机制假说。

**工作流程**：
1. 加载 cisTarget ranking 数据库（全基因组基序扫描预计算结果）
2. 加载 motif 注释（TF-基序对应关系表）
3. 对每个 TF，在 ranking 数据库中做靶基因集合的基序富集分析
4. 富集显著的 target gene set 保留为 regulon

**为什么需要数据库？** 基序分析需要全基因组的基序扫描结果（每个基因的
启动子/增强子区域有多少个哪种 TF 的结合基序）。这些数据从参考基因组和
motif 数据库预计算而来，结果即为 cisTarget ranking 数据库。

**下载指引**：
1. 访问 [SCENIC resources](https://resources.aertslab.org/cistarget/)
2. 下载 hg38 数据库：
   - `hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather` (~1.5GB)
   - `motifs-v9-nr.hgnc-m0.001-o0.0.tbl` (~200MB)
3. 下载 TF 列表 `allTFs_hg38.txt`
4. 将三个文件放入 `references/cisTarget/`
5. 确保 numpy 兼容性（见上方守卫提示）
6. 修改 PARAMS cell 中路径后重跑

下游 AUCell 步骤依赖 cisTarget 的输出——因此 cisTarget 不可用时
AUCell 也一并跳过。


In [ ]:
# === Step 2: cisTarget 模块修剪 ===
# 加载 cisTarget 排名数据库，对每个 TF 的候选靶基因做顺式调控基序富集分析。

_cis_output_dir = "results/cisTarget_output"
_cis_adjacencies_path = os.path.join(_cis_output_dir, "adjacencies.csv")
_cis_regulons_path = os.path.join(_cis_output_dir, "regulons.csv")

if not _cistarget_ready:
    print("cisTarget 条件不满足（数据库缺失或 numpy 不兼容），跳过模块修剪。")
    print("修复后重跑本 notebook 即可启用。")
    _cistarget_done = False
elif len(_links_df) == 0:
    print("WARNING: GRNBoost2 未产出任何链接，跳过 cisTarget。")
    _cistarget_done = False
else:
    # 此时 _cistarget_ready=True，所有 import 已在守卫 cell 中成功
    from pyscenic.prune import prune2df
    from pyscenic.cli.utils import load_signatures
    from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase

    os.makedirs(_cis_output_dir, exist_ok=True)

    # 2a. 加载 ranking 数据库
    print(f"加载 ranking 数据库: {CISTARGET_RANKINGS_PATH}")
    _ranking_dbs = [RankingDatabase(fname=CISTARGET_RANKINGS_PATH)]
    print(f"  ranking DB 物种: {_ranking_dbs[0].name}")

    # 2b. 加载 motif 注释
    print(f"加载 motif 注释: {CISTARGET_MOTIF_PATH}")
    _motif_annotations = load_signatures(CISTARGET_MOTIF_PATH)
    print(f"  motif 数: {len(_motif_annotations)}")

    # 2c. 加载 TF 列表
    print(f"加载 TF 列表: {CISTARGET_TF_LIST_PATH}")
    with open(CISTARGET_TF_LIST_PATH) as _f:
        _tf_list = [line.strip() for line in _f if line.strip()]
    print(f"  TF 数: {len(_tf_list)}")

    # 2d. 过滤 GRNBoost2 链接：仅保留 TF 在已知 TF 列表中的链接
    _links_for_cis = _links_df[_links_df["TF"].isin(_tf_list)]
    _pct_kept = len(_links_for_cis) / max(len(_links_df), 1) * 100
    print(f"已知 TF 过滤后链接数: {len(_links_for_cis):,} / "
          f"{len(_links_df):,} ({_pct_kept:.1f}%)")

    if len(_links_for_cis) > 0:
        _links_for_cis.to_csv(_cis_adjacencies_path, index=False)
        print(f"Adjacencies 已保存: {_cis_adjacencies_path}")

        # 2e. cisTarget 修剪——对每个 TF 做基序富集分析
        print(f"正在执行 cisTarget 模块修剪 "
              f"(n_workers={CISTARGET_NUM_WORKERS})...")
        _modules = list(
            prune2df(
                _ranking_dbs,
                _links_for_cis,
                _motif_annotations,
                num_workers=CISTARGET_NUM_WORKERS,
            )
        )
        print(f"cisTarget 完成: {len(_modules)} 个 regulon")

        # 2f. 导出 regulon 表格
        from pyscenic import export2df
        _regulon_df = export2df(_modules)
        _regulon_df.to_csv(_cis_regulons_path, index=False)
        print(f"Regulons 已保存: {_cis_regulons_path}")
        _cistarget_done = True
    else:
        print("WARNING: TF 列表与数据集基因交集为空，跳过 cisTarget。")
        print("  请检查 TF 列表的基因符号是否与本数据集的 gene symbol 一致。")
        _cistarget_done = False


## 3. AUCell -- Regulon 活性打分

**为什么做 AUCell？** cisTarget 产物是"regulon"（每个 TF 的修剪后靶基因
集合）——但这是全数据集的静态定义，不能反映"不同细胞状态中哪个 TF 真正
在发挥作用"。AUCell 为每个细胞计算每个 regulon 的活性分数——本质上是
问"这个细胞表达的基因中，有多少属于 regulon X 的靶基因集合？"

**AUCell 算法**（Area Under the Curve）：
1. 对每个细胞，按其基因表达量从高到低排序
2. 检查 regulon 的靶基因在这个排序列表中的位置分布
3. 如果靶基因倾向于出现在排序顶部 → AUC 高 → TF 在该细胞中活跃
4. 如果靶基因均匀分布于排序中 → AUC 低 → TF 在该细胞中不活跃

**为什么主控转录因子的活性评分是下游分析的核心？** 一个 regulon
的活性高意味着该 TF 在该细胞中正在驱动其靶基因的表达——这是一个
可以量化的"转录调控状态"指标。这些评分矩阵可以直接用作：
- **跨簇比较**：哪些 regulon 在特定细胞簇中特异性高活性？
- **跨条件比较**：疾病 vs 正常中哪些 regulon 活性变化最大？
- **伪时间分析**：发育/疾病进展过程中 regulon 活性的动态变化
- **调控层级重建**：上游 master TF 活性变化如何传递到下游 effector

AUCell 分数写入 `adata.obsm['X_regulon_auc']`（细胞 x regulon 矩阵），
可直接用作下游分析的输入（聚类、UMAP、跨簇比较）。

> **为什么用 counts 排序？** pySCENIC 推荐 AUCell 使用原始计数来排名基因，
> 因为计数是"表达证据的直接度量"。如果 counts layer 不可用，回退到 adata.X。

### Regulon 活性可视化（紧随 AUCell 计算）

对每个细胞的 regulon 活性评分进行可视化，揭示哪些转录因子在
不同细胞群体中活跃：

- **Regulon 跨簇热图**：各 Leiden 簇的 regulon 平均活性（Z-score 归一化），
  取跨簇方差最大的 regulon 展示——差异越大的 regulon 越可能是簇的特征调控因子
- **Top regulon UMAP**：在 UMAP 上按单个 regulon 的 AUC 活性着色，
  直观定位每个主控转录因子活跃的细胞群体

> 以下可视化仅在 cisTarget + AUCell 完成后产出；若跳过则输出提示信息。


In [ ]:
# === Step 3: AUCell regulon 活性打分 ===
# 对每个细胞计算每个 regulon 的 AUC 活性分数。
# 依赖 cisTarget 步骤的产出——如果 cisTarget 未运行则跳过。

if not _cistarget_done:
    print("cisTarget 步骤未完成——跳过 AUCell regulon 活性打分。")
    print("AUCell 需要 cisTarget 输出的 regulon 定义。")
    _aucell_done = False
elif not os.path.exists(_cis_regulons_path):
    print(f"cisTarget 输出文件不存在: {_cis_regulons_path}——跳过 AUCell。")
    _aucell_done = False
else:
    from pyscenic.aucell import aucell

    print(f"加载 regulons: {_cis_regulons_path}")
    _regulons = pd.read_csv(_cis_regulons_path)

    # AUCell 需要计数矩阵来排序基因——优先 counts layer
    if "counts" in adata.layers:
        _auc_expr = adata.layers["counts"]
    elif adata.raw is not None:
        _auc_expr = adata.raw.X
    else:
        _auc_expr = adata.X

    # 转为 dense（AUCell 需要 dense 矩阵做逐行排序）
    if sp.issparse(_auc_expr):
        _auc_mat = _auc_expr.toarray()
    else:
        _auc_mat = np.asarray(_auc_expr)
    print(f"AUCell 输入矩阵: {_auc_mat.shape[0]} 细胞 x {_auc_mat.shape[1]} 基因")

    # 运行 AUCell
    print(f"正在执行 AUCell (n_workers={AUC_NUM_WORKERS})...")
    _auc_mtx = aucell(
        _auc_mat,
        _regulons,
        num_workers=AUC_NUM_WORKERS,
    )
    print(f"AUCell 完成: {_auc_mtx.shape[0]} 细胞 x {_auc_mtx.shape[1]} regulons")

    # 写入 adata.obsm
    adata.obsm["X_regulon_auc"] = _auc_mtx.astype(np.float32)

    # 写入 regulon 名称
    adata.uns["regulon_names"] = list(
        _regulons.columns
        if hasattr(_regulons, "columns")
        else [f"regulon_{i}" for i in range(_auc_mtx.shape[1])]
    )
    print(f"Regulon AUC 矩阵已写入 adata.obsm['X_regulon_auc']")
    print(f"  shape: {adata.obsm['X_regulon_auc'].shape}")
    _aucell_done = True


In [ ]:
# === Regulon AUC 可视化（仅当 AUCell 已完成） ===

_aucell_done_flag = "_aucell_done" in dir() and _aucell_done

if _aucell_done_flag and "X_regulon_auc" in adata.obsm:
    _auc_data = adata.obsm["X_regulon_auc"]
    _regulon_names = adata.uns.get(
        "regulon_names",
        [f"regulon_{i}" for i in range(_auc_data.shape[1])]
    )

    # Fig 2: Regulon 跨簇热图（Z-score 归一化）
    if LEIDEN_COL in adata.obs.columns:
        _clusters = sorted(adata.obs[LEIDEN_COL].astype(str).unique())
        _auc_per_cluster = pd.DataFrame(
            index=_regulon_names,
            columns=_clusters,
            dtype=np.float32,
        )
        for _clu in _clusters:
            _mask = adata.obs[LEIDEN_COL].astype(str) == _clu
            if _mask.sum() > 0:
                _auc_per_cluster[_clu] = _auc_data[_mask.values].mean(axis=0)

        # 取 variance 最大的 top 20 regulon
        _auc_var = _auc_per_cluster.var(axis=1).sort_values(ascending=False)
        _top_regulons = _auc_var.head(min(20, len(_regulon_names))).index.tolist()

        if len(_top_regulons) > 0:
            _auc_plot = _auc_per_cluster.loc[_top_regulons]
            _auc_plot_z = _auc_plot.subtract(
                _auc_plot.mean(axis=1), axis=0
            ).divide(_auc_plot.std(axis=1) + 1e-12, axis=0)

            fig, ax = plt.subplots(
                figsize=(max(6, len(_clusters) * 0.6),
                         max(4, len(_top_regulons) * 0.35))
            )
            sns.heatmap(
                _auc_plot_z,
                cmap="RdBu_r", center=0,
                xticklabels=True, yticklabels=True,
                linewidths=0.3,
                cbar_kws={"label": "Regulon AUC (Z-score)"},
                ax=ax,
            )
            ax.set_title("Top Regulon Activity across Clusters (Z-score)")
            ax.set_xlabel("Cluster")
            ax.set_ylabel("Regulon")
            plt.tight_layout()
            _heatmap_path = "results/figures/13_grn_regulon_heatmap.png"
            fig.savefig(_heatmap_path, dpi=200, bbox_inches="tight")
            plt.close("all")
            print(f"Regulon 跨簇热图已保存: {_heatmap_path}")

    # 图 3：Top 4 regulon AUC 活性 UMAP 可视化
    if "X_umap" in adata.obsm:
        _auc_var = _auc_per_cluster.var(axis=1).sort_values(ascending=False)
        _top4 = _auc_var.head(min(4, len(_regulon_names))).index.tolist()
        _umap = adata.obsm["X_umap"]
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for _i, _reg in enumerate(_top4):
            _reg_idx = (
                _regulon_names.index(_reg)
                if _reg in _regulon_names
                else _i
            )
            _ax = axes[_i]
            _sc = _ax.scatter(
                _umap[:, 0], _umap[:, 1],
                c=_auc_data[:, _reg_idx],
                cmap="viridis", s=3, alpha=0.8, rasterized=True,
            )
            plt.colorbar(_sc, ax=_ax, label="AUC")
            _ax.set_title(f"Regulon AUC: {_reg}")
            _ax.set_xlabel("UMAP 1")
            _ax.set_ylabel("UMAP 2")

        for _i in range(len(_top4), 4):
            axes[_i].set_visible(False)

        plt.tight_layout()
        _umap_path = "results/figures/13_grn_regulon_umap.png"
        fig.savefig(_umap_path, dpi=200, bbox_inches="tight")
        plt.close("all")
        print(f"Regulon AUC UMAP 已保存: {_umap_path}")
else:
    print("AUCell 未完成——跳过 regulon 可视化。")
    print("完成 cisTarget 数据库下载 + numpy 兼容性修复 + 重跑后即可产出 regulon 图。")


In [ ]:
# 运行摘要与元数据写入 adata.uns。
import datetime as _dt

_13_grn_uns = {
    "method": "pySCENIC (GRNBoost2 + cisTarget + AUCell)",
    "leiden_col": LEIDEN_COL,
    "n_hvg_used": _n_hvg_use,
    "n_links_grnboost2": len(_links_df),
    "cistarget_db_ready": _cistarget_db_ready,
    "cistarget_importable": _cistarget_importable,
    "cistarget_ran": _cistarget_done if "_cistarget_done" in dir() else False,
    "aucell_ran": _aucell_done if "_aucell_done" in dir() else False,
    "timestamp": _dt.datetime.now().isoformat(),
}

_13_grn_uns["cistarget_resources"] = {
    "rankings_path": CISTARGET_RANKINGS_PATH,
    "motif_path": CISTARGET_MOTIF_PATH,
    "tf_list_path": CISTARGET_TF_LIST_PATH,
    "resources_ready": _cistarget_db_ready,
    "numpy_compat_ok": _cistarget_importable,
}

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 命名一致）
adata.uns["stage"] = "13_grn"     # 本 stage 标识
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"


adata.uns["13_grn_v1"] = _13_grn_uns
print("运行元数据已写入 adata.uns['13_grn_v1']")
print()
print("=" * 50)
print("07 GRN 执行摘要:")
print(f"  GRNBoost2: 已完成 ({len(_links_df):,} TF-target links, "
      f"{len(_hvg_genes)} HVG genes)")
print(f"  cisTarget: {'已完成' if _cistarget_done else '跳过'}"
      f" (DB={'Y' if _cistarget_db_ready else 'N'}, "
      f"numpy={'OK' if _cistarget_importable else 'FAIL'})")
print(f"  AUCell:    {'已完成' if ('_aucell_done' in dir() and _aucell_done) else '跳过'}"
      f" (依赖 cisTarget)")
print("=" * 50)


In [ ]:
# 内存自检——确保 X 没有被误转为 dense。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增数据
if "X_regulon_auc" in adata.obsm:
    _auc_shape = adata.obsm["X_regulon_auc"].shape
    _auc_dtype = adata.obsm["X_regulon_auc"].dtype
    print(f"  X_regulon_auc: {_auc_shape}, dtype={_auc_dtype}")
    print(f"  regulon 数: {len(adata.uns.get('regulon_names', []))}")
else:
    print("  X_regulon_auc: 未写入（cisTarget/AUCell 未运行或已跳过）")


In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")
